In [1]:
import os, sys
sys.path.append(os.path.abspath("F:/Neutrino_SI/Pycode"))  # 让 Python 能找到该目录的模块

In [ ]:
import numpy as np
import SignalRate
from scipy import integrate
import Data
import definition as df
import Errors
from scipy import optimize
import emcee
import matplotlib.pyplot as plt
import corner
import multiprocessing as mp
from multiprocessing.dummy import Pool as ThreadPool


### setup ↓

# Range of triple integral of three detectors, from left to right the range of time, energy and cosθ.

ranges_K = [(0, 30), (4, 100), (-1, 1)]     

ranges_I = [(0, 30), (19, 100), (-1, 1)]

ranges_B = [(0, 30), (10, 100), (-1, 1)]

initial = np.array([5, 5, 2, 0.5, 1, 1])

R_fix = np.arange(12, 21, 1)

M_fix = 0.3

# best fit 6p: [12.597, 5.036, 4.688, 0.291, 2.179, 0.527]
percision = 16      # percision of self-defined triple integral, no smaller than 16

# labels = ['$R_c$', '$T_c$', r'$\tau_c$' , '$M_a$','$T_a$', r'$\tau_a$', r'$m_{\Phi}$', r'${\lambda}_{\Phi \nu}$']

labels = [r'$T_c$', r'$\tau_c$' , r'$T_a$', r'$\tau_a$', r'log($m_{\Phi}$)/eV', r'log(${\lambda}_{\Phi \nu}/10^{-8}$)']      # labels of parameters (copy above)

### setup ↑

(t_K, E_K, c_K, dE_K, B_K
 , t_I, E_I, c_I, dE_I, B_I
 , t_B, E_B, c_B, dE_B, B_B) = (Data.Kam_t, Data.Kam_E, Data.Kam_c, Data.Kam_dE, Data.Kam_B_2009
                                , Data.IMB_t, Data.IMB_E, Data.IMB_c, Data.IMB_dE, Data.IMB_B
                                , Data.Baksan_t, Data.Baksan_E, Data.Baksan_c, Data.Baksan_dE, Data.Baksan_B)


DATA = None   # 大数组和常量
CFG  = None   # 可变的小配置（本轮选择的 i 等）

def init_worker(data, cfg):
    """在每个子进程启动时调用，一次性把大数据放进全局。"""
    global DATA, CFG
    DATA = data
    CFG  = cfg

#Kamiokande
def log_likelihood_K(theta):

    # R_c, T_c, tau_c, M_a, T_a, tau_a, m_phi, lambda_nu = theta

    R_fix = DATA

    i=CFG.x

    T_c, tau_c, T_a, tau_a, m_phi, lambda_nu = theta

    fun1_K = lambda x: SignalRate.SR_K(t_K, x, c_K, *[R_fix[i], T_c, tau_c, M_fix, T_a, tau_a, m_phi, lambda_nu]) * Errors.Error_E(x, E_K, dE_K)        # optional

    part1_K = df.gl3_integrate(SignalRate.SR_K, ranges_K, percision, args=[R_fix[i], T_c, tau_c, M_fix, T_a, tau_a, m_phi, lambda_nu])

    part2_K = integrate.quad_vec(fun1_K, 4, 100)[0]     # optional

    part3_K = B_K/2

    ret_K = -part1_K + np.sum(np.log(part3_K +part2_K))

    return ret_K*1e3 


# IMB
def log_likelihood_I(theta):

    R_fix = DATA

    i=CFG.x

    T_c, tau_c, T_a, tau_a, m_phi, lambda_nu = theta
    
    fun1_I = lambda x: SignalRate.SR_I(t_I, x, c_I, *[R_fix[i], T_c, tau_c, M_fix, T_a, tau_a, m_phi, lambda_nu])*Errors.Error_E(x, E_I, dE_I)

    fun2_I = lambda x, c, t: SignalRate.SR_I(t, x, c, *[R_fix[i], T_c, tau_c, M_fix, T_a, tau_a, m_phi, lambda_nu])*Errors.Error_E(x, E_I, dE_I)


    part1_I = df.gl3_integrate(SignalRate.SR_I, ranges_I, percision, args=[R_fix[i], T_c, tau_c, M_fix, T_a, tau_a, m_phi, lambda_nu])


    # ******************************


    part2_I = np.clip(integrate.quad_vec(fun1_I, 19 , 100)[0], 1e-100, None)

    part3_I = B_I/2

    part4_I= df.gl2_integrate_vec(fun2_I, ((19,100), (-1,1)), t_I, percision)

    ret_I = -0.9055*part1_I  - np.sum(0.035*part4_I) + np.sum(np.log(part3_I + part2_I))

    return ret_I*1e3


# Baksan
def log_likelihood_B(theta):

    R_fix = DATA

    i=CFG.x

    T_c, tau_c, T_a, tau_a, m_phi, lambda_nu = theta

    fun1_B = lambda x: SignalRate.SR_B(t_B, x, c_B, *[R_fix[i], T_c, tau_c, M_fix, T_a, tau_a, m_phi, lambda_nu])*Errors.Error_E(x, E_B, dE_B)

    part1_B = df.gl3_integrate(SignalRate.SR_B, ranges_B, percision, args=[R_fix[i], T_c, tau_c, M_fix, T_a, tau_a, m_phi, lambda_nu])

    part2_B = integrate.quad_vec(fun1_B, 10, 100)[0]

    part3_B = B_B/2

    ret_B = - part1_B + np.sum(np.log(part3_B + part2_B))

    return ret_B*1e3


# Total
def log_likelihood(theta):

    ret = (
        log_likelihood_K(theta) +
        #    log_likelihood_I(theta) +
        log_likelihood_B(theta)
        )

    return ret


def log_prior(theta):

    T_c, tau_c, T_a, tau_a, m_phi, lambda_nu = theta
    if 1 < T_c < 10 and 1 < tau_c < 10 and  1 < T_a < 10 and 0.1 < tau_a < 2 and 0 <= m_phi <= 4 and -1 <= lambda_nu <= 3:

        return 0.0
    return -np.inf


def log_prob(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)


def main():
    # ---- 打包大数组与常量：只传一次给每个 worker ----
    data = R_fix

    ctx = mp.get_context("spawn")
    mgr = ctx.Manager()
    cfg = mgr.Namespace()
    cfg.x = 0   # 本轮的 R_fix 索引

    pos = initial + 1e-4 * np.random.randn(25, len(initial))   

    nwalkers, ndim = pos.shape

    # global DATA, CFG
    # DATA = data
    # CFG  = cfg

    with ctx.Pool(processes=min(ctx.cpu_count(), nwalkers),
                  initializer=init_worker, initargs=(data, cfg)) as pool:

        # 只建一次 Sampler；避免每轮重建
        sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob, pool=pool)

        for i in range(len(R_fix)):
            cfg.x = i  # 切换到本轮的 θ 设定（R_fix[i]）

            print('fix R = :', R_fix[i], ', fix M = :', M_fix)

            # print('test result:', log_likelihood(initial)/1e3)

            sampler.run_mcmc(pos, 3000, progress=False)

            flat_samples = sampler.get_chain(discard=2000, thin=5, flat=True)

            path_figure = r"F:/Neutrino_SI/Figures/fixR" +str(R_fix[i]) +"M03_K_B"      

            path_chain = r"F:/Neutrino_SI/Data/fixR" + str(R_fix[i]) +"M03_K_B.xlsx" 

            df.save_mcmc_result(flat_samples, path= path_chain)

            # print("best fit values: ")
            print("best fit for R = " + str(R_fix[i]) + ":")
            print(["{0:.3f}".format(np.percentile(flat_samples[:, j], 50)) for j in range(ndim)])

            plt.figure()
            fig = corner.corner(
                flat_samples, quantiles=[0.16, 0.5, 0.84],show_titles=True, title_kwargs={"fontsize": 12}, smooth = 1, labels=labels,
                plot_datapoints=False
            )

            plt.savefig(path_figure)



if __name__ == "__main__":
    # 建议同时在最开头加：
    # import os; os.environ["OMP_NUM_THREADS"]="1"; os.environ["MKL_NUM_THREADS"]="1"
    main()

fix R = : 12 , fix M = : 0.3
